In [0]:
%run ./Classroom-Setup-Common

In [0]:
########################################
## Course helper: create the per-environment catalogs
##
## Moved here (out of Classroom-Setup-Common-Python) so the common notebook
## stays a clean, general-purpose library. This function reuses the common's
## _catalog_exists() and _get_workspace_catalogs() helpers, which are available
## after the %run above.
########################################

def create_env_catalogs(base_catalog: str, suffixes: list):
    """
    Create per-environment catalogs derived from a base catalog name.

    For a base_catalog of 'labuser_jsmith' and suffixes
    ['_1_dev', '_2_stage', '_3_prod'], this creates:
        labuser_jsmith_1_dev
        labuser_jsmith_2_stage
        labuser_jsmith_3_prod

    The base catalog MUST already be resolved and created (i.e. the return value
    of build_user_catalog(), exposed as my_catalog by the common notebook). If
    base_catalog is missing/empty -- meaning the earlier catalog setup failed --
    or does not actually exist in the workspace, this function stops immediately
    and creates NOTHING.

    Parameters
    ----------
    base_catalog : str
        The resolved base catalog name (the my_catalog value).
    suffixes : list of str
        Suffixes appended to base_catalog, e.g. ['_1_dev', '_2_stage', '_3_prod'].
        Built by the caller and passed in.

    Returns
    -------
    list
        The environment catalog names that now exist.
    """
    # -- Guard 1: no base catalog means the upstream setup failed ----------
    if not base_catalog or not str(base_catalog).strip():
        raise ValueError(
            "❌ No base catalog was provided. build_user_catalog() must succeed first. "
            "Aborting -- no suffixed catalogs will be created."
        )

    base_catalog = base_catalog.strip()

    # -- Guard 2: confirm the base catalog actually exists -----------------
    if not _catalog_exists(base_catalog, _get_workspace_catalogs()):
        raise ValueError(
            f"❌ Base catalog '{base_catalog}' does not exist in this workspace. "
            "Aborting -- no suffixed catalogs will be created."
        )

    # -- Guard 3: suffixes must be a non-empty list of strings -------------
    if not isinstance(suffixes, list) or not suffixes or not all(isinstance(s, str) for s in suffixes):
        raise ValueError(
            "❌ 'suffixes' must be a non-empty list of strings, "
            "e.g. ['_1_dev', '_2_stage', '_3_prod']."
        )

    print(f"\n{'='*60}")
    print(f"  Creating environment catalogs from base: {base_catalog}")
    print(f"{'='*60}")

    existing = set(_get_workspace_catalogs())  # snapshot once before the loop
    created_catalogs = []

    for i, suffix in enumerate(suffixes, start=1):
        env_catalog = f"{base_catalog}{suffix}"
        print(f"  [{i}/{len(suffixes)}] {env_catalog}...", end=" ")
        try:
            if env_catalog.lower() in existing:
                print("ALREADY EXISTS")
            else:
                spark.sql(f"CREATE CATALOG IF NOT EXISTS {env_catalog}")
                print("CREATED")
            created_catalogs.append(env_catalog)
        except Exception as e:
            print("FAILED")
            print(f"      ⚠️ Could not create '{env_catalog}'. You may lack catalog-creation "
                  f"privileges in this workspace.\n      Error: {e}")

    print(f"\n{'='*60}")
    print(f"  COMPLETE: {len(created_catalogs)}/{len(suffixes)} environment catalog(s) ready.")
    print(f"{'='*60}\n")

    return created_catalogs

In [0]:
########################################
## Course data: build the source Spark DataFrame from the CDC diabetes dataset
##
## Course-specific helper (previously a DA method). Plain function now.
##
## Serverless-safe: the deterministic "PII" column is built with NATIVE Spark
## functions (md5 + formatting) instead of a Python UDF. Python UDFs on
## serverless (Spark Connect) can trigger
## [INCOMPATIBLE_CPU_ARCHITECTURE_NOT_SAFE_TO_RESTART], so they are avoided.
########################################

def create_spark_data_frame_from_cdc(cdc_csv_file_path):
    '''
    Create the Spark DataFrame used to generate the course CSV files.

    Reads the raw CDC diabetes CSV, then adds:
      - ID:          a monotonically increasing row id
      - PII:         a deterministic, UUID-shaped value derived from ID
                     (native md5 hash formatted 8-4-4-4-12 -- no Python UDF)
      - date:        a synthetic date bucketed by ID
      - HighCholest: a bucketed (0/1/2) version of HighChol
    and selects the columns needed for the course.

    Parameters
    ----------
    cdc_csv_file_path : str
        Full path to the raw CDC diabetes CSV, e.g.
        /Volumes/dbacademy_cdc_diabetes/v01/cdc-diabetes/diabetes_binary_5050_raw.csv

    Returns
    -------
    pyspark.sql.DataFrame
    '''
    from pyspark.sql.functions import (
        when, lit, monotonically_increasing_id, col, md5, substring, concat_ws
    )

    # Deterministic, UUID-shaped PII derived from ID using native functions only.
    # (Replaces the previous uuid.uuid5 Python UDF, which is not serverless-safe.)
    def _deterministic_uuid(id_col):
        h = md5(id_col.cast("string"))
        return concat_ws(
            "-",
            substring(h, 1, 8),
            substring(h, 9, 4),
            substring(h, 13, 4),
            substring(h, 17, 4),
            substring(h, 21, 12),
        )

    sdf = (spark
        .read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(cdc_csv_file_path)
        .repartition(1)
        .withColumn("ID", monotonically_increasing_id())
        .select("ID",
                _deterministic_uuid(col("ID")).alias('PII'),
                when(col("ID") < 15121, "2025-01-01")
                    .when(col("ID") < 40351, "2025-01-02")
                    .otherwise("2025-01-03")
                    .alias("date"),
                when(col("HighChol") < .2, 0)  # when value is less than .2
                    .when((col("HighChol").cast('float') >= .2) & (col("HighChol") < 1.03), 1)  # between .2 and 1.03
                    .otherwise(2)              # else >= 1.03
                    .alias("HighCholest"),
                "HighBP",
                "BMI",
                "Age",
                "Education",
                "income"
        )
    )

    return sdf

In [0]:
########################################
## Course data: write the CSV files into the main/dev/stage/prod volumes
##
## Course-specific helper (previously a DA method). Plain function now.
## Uses the common notebook's delete_files() in place of DA.delete_source_files().
########################################

def create_course_csv_files(dataframe,
                            prod_volume_write_path: str,
                            stage_volume_write_path: str,
                            dev_volume_write_path: str,
                            main_volume_write_path: str,
                            data_filter_conditions: list,
                            data_name_append: str,
                            del_files_first: bool = True):
    '''
    Create the course CSV files across the main/dev/stage/prod volumes.

    By default first deletes existing files in each target volume path, then writes:
        - main volume  : 1 CSV, subset (ID <= 7499), PII masked
        - dev volume   : 1 CSV, subset (ID <= 7499), PII masked
        - stage volume : 1 CSV, subset (ID <= 34999)
        - prod volume  : 1 CSV per date filter (full data)
    '''
    from pyspark.sql.functions import rand, when, lit, to_date, monotonically_increasing_id, col, coalesce
    import os

    print('\n----Creating the CSV files in the necessary volumes for the course----\n')

    ##
    ## Delete the files in the volumes prior to creating the CSV files.
    ## (delete_files comes from the common notebook — replaces DA.delete_source_files)
    ##
    if del_files_first == True:
        print('---DELETE FILES IN THE HEALTH VOLUME WITHIN THE MAIN, DEV, STAGE, PROD CATALOGS----')
        delete_files(volume_folder_path=prod_volume_write_path)
        delete_files(volume_folder_path=stage_volume_write_path)
        delete_files(volume_folder_path=dev_volume_write_path)
        delete_files(volume_folder_path=main_volume_write_path)

    ## Utility: remove Spark's underscore-prefixed files (_SUCCESS, etc.) after a CSV write.
    def util_del_files_with_underscore_in_volume(volume_path: str):
        files = dbutils.fs.ls(volume_path)
        files_to_remove = [file.path for file in files if file.name.startswith('_')]
        if len(files_to_remove) == 0:
            pass
        else:
            print(f'Deleting files with underscores in: {volume_path}:')
            for file_path in files_to_remove:
                dbutils.fs.rm(file_path)

    ## Utility: rename the part-* CSV to <prefix>_<suffix>.csv
    def util_rename_csv_file(volume_path: str, file_prefix: str, file_suffix: str):
        output_files = os.listdir(volume_path)
        for file in output_files:
            if file.startswith('part-'):
                file_to_rename = f'{volume_path}/{file}'
                new_csv_file_name = f'{file_prefix}_{file_suffix}.csv'
                rename_file_to = f'{volume_path}/{new_csv_file_name}'
                print(f'Renaming CSV file: {file_to_rename}')
                print(f'New CSV file name: {rename_file_to}')
                os.rename(file_to_rename, rename_file_to)
                print(f'Created CSV file: {new_csv_file_name}')
        print('\n')

    ## Prod: one CSV per date filter, full data.
    def create_prod_csv_files():
        print('-------------------------------------------------------')
        print(f'--- Creating CSV files in PROD: {prod_volume_write_path}')
        print('-------------------------------------------------------')
        for filter in data_filter_conditions:
            (dataframe
             .filter(col("date") == filter)
             .repartition(1)   ## One CSV file per date
             .write
             .option("header", "true")
             .mode('append')
             .csv(prod_volume_write_path)
            )
            ## Del files with underscores after writing the csv files
            util_del_files_with_underscore_in_volume(prod_volume_write_path)
            ## Rename file with the filter condition + data_name_append
            util_rename_csv_file(prod_volume_write_path, file_prefix=filter, file_suffix=data_name_append)
        print(f'Finished creating CSV files in PROD: {prod_volume_write_path} \n')

    ## Dev/main: single CSV subset (ID <= 7499), PII masked.
    def create_dev_csv_files(spec_volume_path):
        print('-------------------------------------------------------')
        print(f'--- Creating CSV files in DEV: {spec_volume_path}')
        print('-------------------------------------------------------')
        (dataframe
         .filter(col("ID") <= 7499)
         .repartition(1)   ## One CSV file per date
         .withColumn("PII", lit("********"))
         .write
         .option("header", "true")
         .mode('append')
         .csv(spec_volume_path)
        )
        util_del_files_with_underscore_in_volume(spec_volume_path)
        util_rename_csv_file(spec_volume_path, file_prefix="dev", file_suffix=data_name_append)
        print(f'Finished creating CSV file in DEV: {spec_volume_path} \n')

    ## Stage: single CSV subset (ID <= 34999).
    def create_stage_csv_files():
        print('-------------------------------------------------------')
        print(f'--- Creating CSV files in STAGE: {stage_volume_write_path}')
        print('-------------------------------------------------------')
        (dataframe
         .filter(col("ID") <= 34999)
         .repartition(1)   ## One CSV file per date
         .write
         .option("header", "true")
         .mode('append')
         .csv(stage_volume_write_path)
        )
        util_del_files_with_underscore_in_volume(stage_volume_write_path)
        util_rename_csv_file(stage_volume_write_path, file_prefix="stage", file_suffix=data_name_append)
        print(f'Finished creating CSV file in STAGE: {stage_volume_write_path} \n')

    ## Create the CSV files for MAIN, DEV, STAGE, PROD
    create_dev_csv_files(spec_volume_path=dev_volume_write_path)
    create_dev_csv_files(spec_volume_path=main_volume_write_path)
    create_stage_csv_files()
    create_prod_csv_files()

In [0]:
########################################
## Setup course environment
########################################

## The common notebook (%run above) already resolved the base catalog into `my_catalog`.
## If that failed, there is no point creating the environment catalogs.
if not my_catalog:
    raise RuntimeError("❌ Base catalog setup failed in the common notebook — aborting course setup.")

## Environment suffixes — built here and passed into create_env_catalogs().
list_of_catalog_suffixes = ['_1_dev', '_2_stage', '_3_prod']

## Create the user-specific environment catalogs (idempotent).
create_env_catalogs(my_catalog, list_of_catalog_suffixes)

## Explicit references to each environment catalog (replaces the old DA.catalog_* attributes).
catalog_dev   = f"{my_catalog}_1_dev"
catalog_stage = f"{my_catalog}_2_stage"
catalog_prod  = f"{my_catalog}_3_prod"

## Course schema — a dedicated 'dev_ops' schema is used in every catalog for
## consistency, instead of the generic 'default' schema.
course_schema = 'dev_ops'

## Create the 'dev_ops' schema in each catalog (idempotent).
## create_schemas() comes from Classroom-Setup-Common-Python (available after the %run above).
create_schemas(in_catalog=my_catalog,    schemas_to_create=[course_schema])
create_schemas(in_catalog=catalog_dev,   schemas_to_create=[course_schema])
create_schemas(in_catalog=catalog_stage, schemas_to_create=[course_schema])
create_schemas(in_catalog=catalog_prod,  schemas_to_create=[course_schema])

## Create the 'health' volume inside the 'dev_ops' schema of each catalog.
create_volumes(catalog=my_catalog,    schema=course_schema, volume_names=['health'])
create_volumes(catalog=catalog_dev,   schema=course_schema, volume_names=['health'])
create_volumes(catalog=catalog_stage, schema=course_schema, volume_names=['health'])
create_volumes(catalog=catalog_prod,  schema=course_schema, volume_names=['health'])

##
## Create the CSV files
##

## Build the source Spark DataFrame from the CDC diabetes dataset.
sdf = create_spark_data_frame_from_cdc("/Volumes/dbacademy_cdc_diabetes/v01/cdc-diabetes/diabetes_binary_5050_raw.csv")

## Create the course CSV files across the environment volumes (now in the 'dev_ops' schema).
create_course_csv_files(dataframe=sdf,
                        prod_volume_write_path  = f'/Volumes/{catalog_prod}/{course_schema}/health',
                        stage_volume_write_path = f'/Volumes/{catalog_stage}/{course_schema}/health',
                        dev_volume_write_path   = f'/Volumes/{catalog_dev}/{course_schema}/health',
                        main_volume_write_path  = f'/Volumes/{my_catalog}/{course_schema}/health',
                        data_filter_conditions  = ["2025-01-01", "2025-01-02", "2025-01-03"],
                        data_name_append        = "health",
                        del_files_first         = True)

## Setup Complete
print('\n\n\n------------------------------------------------------------------------------')
print('COURSE SETUP COMPLETE!')
print('------------------------------------------------------------------------------')

## Display the catalog + schema references for the user.
display_config_values(
  [
    ('Main catalog reference: my_catalog', my_catalog),
    ('Dev catalog: catalog_dev', catalog_dev),
    ('Stage catalog: catalog_stage', catalog_stage),
    ('Prod catalog: catalog_prod', catalog_prod),
    ('Course schema (all catalogs)', course_schema),
  ]
)